# Tools and Agents

# Agent: Use LLMs to choose a specific activity to perform
# Tool: Used by agents to perform a specific task

In [ ]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
import os, warnings
from dotenv import load_dotenv

In [ ]:
warnings.filterwarnings("ignore")

env = r"D:\stackroute\2_AI-assisted-programming\learning_requirements\bosch\2026\5_advancedPE\code\config.env"
if load_dotenv(env):
    gemini_key = os.getenv("GEMINI_API_KEY")

In [ ]:
# Tool 1
@tool
def convert_degree_to_fahrenheit(celsius: float) -> float:
    """
    Convert temperature from Celsius to Fahrenheit.
    """
    return (celsius * 9 / 5) + 32


# Tool 2
@tool
def convert_fahrenheit_to_degree(fahrenheit: float) -> float:
    """
    Convert temperature from Fahrenheit to Celsius.
    """
    return (fahrenheit - 32) * 5 / 9

In [ ]:
# 1) Create the Gemini LLM instance
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", api_key=gemini_key, temperature=0)

# 2) Register the tools
tools = [convert_degree_to_fahrenheit, convert_fahrenheit_to_degree]
llm_with_tools = llm.bind_tools(tools)

# 3) Create the user prompt
user_prompt = "Convert 35 degrees Celsius to Fahrenheit."

messages = [HumanMessage(content=user_prompt)]

# 4) Ask Gemini to select the appropriate tool
response = llm_with_tools.invoke(messages)

In [ ]:
# 5) Display the tool-call details
print("Tool calls:")
print(response.tool_calls)

In [ ]:
# 6) Check whether Gemini selected a tool
if not response.tool_calls:
    print("No Tool Selected")
else:
    # Select the first tool call
    tool_call = response.tool_calls[0]

    tool_name = tool_call["name"]
    tool_args = tool_call["args"]

    print("\nSelected tool:", tool_name)
    print("Tool arguments:", tool_args)


    # 7) Create the tool registry
    tool_registry = {tool.name: tool for tool in tools}


    # 8) Execute the selected tool
    selected_tool = tool_registry.get(tool_name)

    if selected_tool is None:
        print("Unknown tool selected:", tool_name)

    else:
        result = selected_tool.invoke(tool_args)

        print("\nTool result:")
        print(result)

In [ ]:
# Example 2) Using Web search tool to get the latest news about a topic

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.agents import create_agent

In [ ]:
# 1) Create the Gemini LLM
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite",api_key=gemini_key,temperature=0)

# 2) Create the Wikipedia tool
wiki_api = WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=1000)
wiki = WikipediaQueryRun(api_wrapper=wiki_api)

tools = [wiki]

# 3) Create the agent
agent = create_agent( model=llm, tools=tools,
    system_prompt="""
    You are a helpful research assistant.

    Use the Wikipedia tool to collect relevant information before
    answering the user's question.

    Provide a concise, factual and easy-to-understand summary.
    """
)

# 4) Prepare the query
content_msg = """
What is the impact of climate change on human health and the environment?
Please provide a concise summary.
"""

In [ ]:
input_message = {"messages": [{"role": "user", "content": content_msg}] }

# 5) Invoke the agent
response = agent.invoke(input_message)

In [ ]:
# 6) Get the final response
response['messages'][-1].content[0]['text']

In [ ]:
# Example 3) Database as a tool to answer questions about a dataset

In [ ]:
import os
import mysql.connector
import pandas as pd
from langchain_core.tools import tool
from langchain.agents import create_agent

In [ ]:
host = "localhost"; database="llyods"; port=3306; user="root"; pwd="root"

# 1) Get MySQL Connection
# -----------------------
def get_mysql_conn():
    try:
        conn = mysql.connector.connect( host=host, port=port, database=database, user=user, password=pwd)
    except Exception as e:
        conn = str(e)

    return(conn)

conn = get_mysql_conn()
print(conn)

In [ ]:
# 3. SQL Tool Definitions
# -----------------------------
@tool
def user_access_tool():
    ''' Use for:
    - failed logins
    - suspicious login behavior
    - privileged users
    - non-business hours access
    - authentication issues
    '''
    query = """
    SELECT * FROM data_security  WHERE failed_logins_24h > 3 AND department = 'Risk';
    """
    return (query)

@tool
def risk_threat_scoring_tool():
    ''' Use for:
    - risk score
    - threat score
    - high-risk users
    - department-wise risk
    - average risk
    '''
    query = """
    SELECT department, AVG(risk_score) AS avg_risk FROM data_security GROUP BY department ORDER BY avg_risk DESC;
    """
    return (query)

@tool
def data_exfiltration_tool():
    ''' Use for:
    - DLP violations
    - customer financial records
    - data sensitivity
    - data transfer
    - data leakage
    - exfiltration
    '''
    query = """
    SELECT user_id, device_type, data_sensitivity_level FROM data_security WHERE data_sensitivity_level > 4;
    """
    return (query)

@tool
def device_endpoint_compliance_tool():
    ''' Use for:
    - unmanaged devices
    - endpoint compliance
    - encryption
    - non-compliant devices
    - device security
    '''
    query = """
    SELECT device_type, encryption_in_transit_flag, unmanaged_device_flag
    FROM data_security
    WHERE unmanaged_device_flag = 0;
    """
    return (query)

@tool
def fallback_sql_tool():
    ''' If none of the above tools are relevant, return a default query result. '''

    query = """ SELECT * FROM data_security LIMIT 10; """
    return (query)

In [ ]:
# User query
# ---------------------------------------------------------

user_prompt = "Show non-compliant endpoints connected to confidential banking resources."

# Tool-selection prompt
# ---------------------------------------------------------

prompt = f'''
            You are an expert in identifying the right tool for a given user query.

            You have access to the following tools:
            1. user_access_tool
            Use for: 
            - failed logins
            - suspicious login behavior
            - privileged users
            - non-business hours access
            - authentication issues

            2. risk_threat_scoring_tool
            Use for:
            - risk score
            - threat score
            - high-risk users
            - department-wise risk
            - average risk

            3. data_exfiltration_tool
            Use for:
            - DLP violations
            - customer financial records
            - data sensitivity
            - data transfer
            - data leakage
            - exfiltration

            4. device_endpoint_compliance_tool
            Use for:
            - unmanaged devices
            - endpoint compliance
            - encryption
            - non-compliant devices
            - device security

            5. fallback_tool
            Use for:
            - If none of the above tools are relevant, return a default query result.

        Instructions:
        - Carefully understand the user's intention.
        - Select only the most appropriate tool.
        - Extract all required arguments from the user query.
        - Convert amounts written with commas into numerical values.
        - Do not invent any missing values.
        - Do not guess the tool name.
        - Do not perform the calculation yourself.
        - If no appropriate tool is available, respond exactly with: No Tool Selected

        User Query:
        {user_prompt}
        '''

In [ ]:
# 1) Create the Gemini LLM instance
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", api_key=gemini_key, temperature=0)

# 2) Register the tools
tools = [user_access_tool, risk_threat_scoring_tool, data_exfiltration_tool, device_endpoint_compliance_tool, fallback_sql_tool]
llm_with_tools = llm.bind_tools(tools)

# 3) Register the user prompt
messages = [HumanMessage(content=prompt)]

# 4) Ask Gemini to select the appropriate tool
response = llm_with_tools.invoke(messages)

In [ ]:
print(response.tool_calls)